# Sistema de Recomendacion de Centros de Salud Preventiva Basado en Capacidad Operativa e Indicadores Climaticos para la Region Puno

**UNAP - Escuela Profesional de Ingenieria de Sistemas - X Semestre**

Este notebook documenta la carga, limpieza, analisis exploratorio, construccion, evaluacion y serializacion de un recomendador preventivo para establecimientos de salud de Puno. Los datos incluidos son reproducibles y equivalentes a los campos publicos RENIPRESS/SUSALUD y SENAMHI requeridos para ejecucion local.

## 1. Instalacion y configuracion

In [ ]:
# Ejecutar si el entorno no tiene las librerias instaladas.
# %pip install pandas scikit-learn haversine folium matplotlib seaborn joblib pyarrow
# %pip install scikit-surprise

from pathlib import Path
import random
import warnings

import joblib
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from folium.plugins import HeatMap
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer, OneHotEncoder

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
print(f'Directorio de trabajo: {BASE_DIR}')

## 2. Carga de datos

In [ ]:
df_renipress = pd.read_csv(DATA_DIR / 'renipress_puno.csv', encoding='utf-8')
print('RENIPRESS shape:', df_renipress.shape)
display(df_renipress.dtypes)
display(df_renipress.head(10))

df_senamhi = pd.read_csv(DATA_DIR / 'senamhi_alertas_puno.csv')
df_senamhi['fecha'] = pd.to_datetime(df_senamhi['fecha'])
print('SENAMHI shape:', df_senamhi.shape)
display(df_senamhi.head(5))

## 3. Analisis Exploratorio de Datos (EDA)

In [ ]:
before_shape = df_renipress.shape
df_clean = df_renipress.drop_duplicates(subset='codigo_renipress').copy()
df_clean['horario'] = df_clean['horario'].fillna('No especificado')
df_clean['categoria'] = df_clean['categoria'].astype(str).str.strip().str.upper()
df_clean['servicios'] = df_clean['servicios'].astype(str).str.lower().str.strip()
df_clean['servicios_lista'] = df_clean['servicios'].str.split('|')
df_clean['servicios_lista'] = df_clean['servicios_lista'].apply(lambda items: [x.strip() for x in items if x.strip()])
df_clean = df_clean[df_clean['latitud'].between(-17, -13) & df_clean['longitud'].between(-71, -68)]
after_shape = df_clean.shape
print(f'Shape antes: {before_shape}; despues: {after_shape}')
display(df_clean[['codigo_renipress', 'nombre', 'categoria', 'provincia', 'servicios_lista']].head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
df_clean['categoria'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#1B4F8A', title='Establecimientos por categoria')
df_clean['provincia'].value_counts().plot(kind='bar', ax=axes[1], color='#2E7D32', title='Establecimientos por provincia')
servicios_series = df_clean.explode('servicios_lista')['servicios_lista'].value_counts().head(10)
servicios_series.plot(kind='bar', ax=axes[2], color='#EA580C', title='Top servicios ofrecidos')
plt.tight_layout()
plt.show()

In [ ]:
categoria_colores = {'I-1': 'gray', 'I-2': 'blue', 'I-3': 'green', 'I-4': 'orange', 'II-1': 'red', 'II-2': 'purple'}
m = folium.Map(location=[-15.84, -70.02], zoom_start=8)
for _, row in df_clean.iterrows():
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=6,
        color=categoria_colores.get(row['categoria'], 'black'),
        fill=True,
        popup=f"{row['nombre']} - {row['categoria']}"
    ).add_to(m)
m.save('mapa_eda.html')
m

In [ ]:
mlb_tmp = MultiLabelBinarizer()
servicios_matrix = pd.DataFrame(mlb_tmp.fit_transform(df_clean['servicios_lista']), columns=mlb_tmp.classes_)
no_nulos = servicios_matrix.to_numpy().sum()
sparsity = 1 - (no_nulos / (servicios_matrix.shape[0] * servicios_matrix.shape[1]))
print(f'Sparsity: {sparsity:.2%}')
plt.figure(figsize=(10, 5))
sns.heatmap(servicios_matrix.head(50), cmap='YlGnBu', cbar=False)
plt.title('Matriz servicios x establecimientos')
plt.xlabel('Servicios')
plt.ylabel('Establecimientos')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_senamhi['nivel_alerta'].value_counts().sort_index().plot(kind='bar', color='#DC2626', ax=ax)
ax.set_title('Distribucion de alertas SENAMHI por nivel')
ax.set_xlabel('Nivel de alerta')
ax.set_ylabel('Cantidad')
plt.show()

heat_map = folium.Map(location=[-15.5, -70.0], zoom_start=8)
heat_data = df_senamhi[['estacion_lat', 'estacion_lon', 'nivel_alerta']].values.tolist()
HeatMap(heat_data, radius=25, blur=18).add_to(heat_map)
heat_map.save('mapa_alertas_senamhi.html')
heat_map

## 4. Construccion del modelo

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return float(2 * r * np.arcsin(np.sqrt(a)))

def build_item_matrix(df_estab, df_alertas):
    df_items = df_estab.merge(df_alertas[['ubigeo', 'nivel_alerta', 'temp_minima', 'precipitacion_mm']], on='ubigeo', how='left')
    df_items['nivel_alerta'] = df_items['nivel_alerta'].fillna(0).astype(int)
    df_items['penalidad'] = df_items['nivel_alerta'] * 0.3
    df_items['capacidad_camas'] = df_items['capacidad_camas'].fillna(0).astype(float)
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    categoria_matrix = ohe.fit_transform(df_items[['categoria']])
    mlb = MultiLabelBinarizer()
    servicios_matrix = mlb.fit_transform(df_items['servicios_lista'])
    scaler = MinMaxScaler()
    numeric = scaler.fit_transform(df_items[['capacidad_camas', 'nivel_alerta']])
    item_matrix = np.hstack([categoria_matrix, servicios_matrix, numeric, (1 - df_items[['penalidad']]).to_numpy()])
    feature_names = list(ohe.get_feature_names_out(['categoria'])) + list(mlb.classes_) + ['capacidad_norm', 'alerta_norm', 'factor_clima']
    return df_items.reset_index(drop=True), item_matrix, scaler, mlb, ohe, feature_names

df_items, item_matrix, scaler, mlb, ohe, feature_names = build_item_matrix(df_clean, df_senamhi)
print('Matriz de items:', item_matrix.shape)
display(pd.DataFrame(item_matrix, columns=feature_names).head())

In [ ]:
def build_user_vector(perfil_usuario, feature_names):
    vector = np.zeros(len(feature_names), dtype=float)
    servicio = perfil_usuario['tipo_atencion'].lower().strip()
    if servicio in feature_names:
        vector[feature_names.index(servicio)] = 1.0
    vector[feature_names.index('capacidad_norm')] = min(perfil_usuario['edad'] / 120, 1)
    vector[feature_names.index('alerta_norm')] = min(perfil_usuario['max_distancia_km'] / 100, 1)
    vector[feature_names.index('factor_clima')] = 1.0
    return vector

def recomendar(perfil_usuario, df_items, item_matrix, n=5):
    df_eval = df_items.copy()
    df_eval['distancia_km'] = df_eval.apply(
        lambda row: haversine_km(perfil_usuario['lat'], perfil_usuario['lon'], row['latitud'], row['longitud']), axis=1
    )
    df_eval = df_eval[df_eval['distancia_km'] <= perfil_usuario['max_distancia_km']].copy()
    if df_eval.empty:
        return pd.DataFrame(columns=['rank', 'nombre', 'categoria', 'distancia_km', 'score_base', 'score_final', 'nivel_alerta'])
    idx = df_eval.index.to_list()
    user_vector = build_user_vector(perfil_usuario, feature_names)
    scores = cosine_similarity([user_vector], item_matrix[idx])[0]
    df_eval['score_base'] = scores
    df_eval['score_final'] = df_eval['score_base'] * (1 - df_eval['penalidad'])
    result = df_eval.sort_values('score_final', ascending=False).head(n).copy()
    result.insert(0, 'rank', range(1, len(result) + 1))
    return result[['rank', 'codigo_renipress', 'nombre', 'categoria', 'provincia', 'distancia_km', 'score_base', 'penalidad', 'score_final', 'nivel_alerta', 'servicios']]

print('Funcion recomendar lista')

In [ ]:
casos = [
    {'nombre': 'Puno capital - control prenatal', 'lat': -15.840221, 'lon': -70.021881, 'tipo_atencion': 'control prenatal', 'edad': 25, 'max_distancia_km': 20},
    {'nombre': 'Moho rural - vacunacion infantil', 'lat': -15.36178, 'lon': -69.49924, 'tipo_atencion': 'vacunacion', 'edad': 2, 'max_distancia_km': 15},
    {'nombre': 'Azangaro con alerta roja - tamizaje', 'lat': -14.90843, 'lon': -70.19608, 'tipo_atencion': 'tamizaje', 'edad': 60, 'max_distancia_km': 30},
]
for caso in casos:
    print('\n' + caso['nombre'])
    display(recomendar(caso, df_items, item_matrix, n=5))

## 5. Evaluacion del modelo

In [ ]:
interacciones = []
for user_id in range(1, 41):
    servicio = random.choice(list(mlb.classes_))
    perfil = {'lat': -15.84 + np.random.normal(0, 0.5), 'lon': -70.02 + np.random.normal(0, 0.5), 'tipo_atencion': servicio, 'edad': random.randint(1, 80), 'max_distancia_km': random.choice([20, 30, 50, 80])}
    recs = recomendar(perfil, df_items, item_matrix, n=5)
    for _, rec in recs.iterrows():
        rating = 3.0 + min(rec['score_final'] * 2, 2)
        interacciones.append((str(user_id), str(rec['codigo_renipress']), float(rating)))
df_interacciones = pd.DataFrame(interacciones, columns=['user_id', 'item_id', 'rating'])
print('Interacciones sinteticas etiquetadas:', df_interacciones.shape)

try:
    from surprise import Dataset, Reader, SVD, BaselineOnly
    from surprise.model_selection import cross_validate
    reader = Reader(rating_scale=(1, 5))
    data = Dataset.load_from_df(df_interacciones[['user_id', 'item_id', 'rating']], reader)
    svd_metrics = cross_validate(SVD(random_state=42), data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
    rmse = float(np.mean(svd_metrics['test_rmse']))
    mae = float(np.mean(svd_metrics['test_mae']))
except Exception as exc:
    rmse = float(np.sqrt(np.mean((df_interacciones['rating'] - df_interacciones['rating'].mean()) ** 2)))
    mae = float(np.mean(np.abs(df_interacciones['rating'] - df_interacciones['rating'].mean())))
    print('Surprise no disponible; se usa baseline promedio reproducible:', exc)

metricas = pd.DataFrame([
    {'modelo': 'Content-Based Filtering', 'RMSE': np.nan, 'MAE': np.nan, 'nota': 'Modelo de ranking sin rating explicito real'},
    {'modelo': 'SVD/Baseline sintetico', 'RMSE': rmse, 'MAE': mae, 'nota': 'Interacciones sinteticas para referencia'},
])
display(metricas)

In [ ]:
def precision_at_k(recomendados, relevantes, k):
    recomendados_k = list(recomendados)[:k]
    return len(set(recomendados_k) & set(relevantes)) / max(k, 1)

def recall_at_k(recomendados, relevantes, k):
    recomendados_k = list(recomendados)[:k]
    return len(set(recomendados_k) & set(relevantes)) / max(len(relevantes), 1)

relevantes = df_items[df_items['servicios'].str.contains('vacunacion')]['codigo_renipress'].astype(str).tolist()
perfil_eval = {'lat': -15.84, 'lon': -70.02, 'tipo_atencion': 'vacunacion', 'edad': 30, 'max_distancia_km': 100}
recomendados = recomendar(perfil_eval, df_items, item_matrix, n=10)['codigo_renipress'].astype(str).tolist()
ks = [1, 3, 5, 10]
df_pr = pd.DataFrame({'k': ks, 'precision': [precision_at_k(recomendados, relevantes, k) for k in ks], 'recall': [recall_at_k(recomendados, relevantes, k) for k in ks]})
display(df_pr)
df_pr.plot(x='k', y='precision', kind='bar', color='#1B4F8A', legend=False, title='Precision@K')
plt.ylim(0, 1)
plt.show()

In [ ]:
perfil_rojo = {'lat': -14.90843, 'lon': -70.19608, 'tipo_atencion': 'tamizaje', 'edad': 60, 'max_distancia_km': 100}
comparacion = recomendar(perfil_rojo, df_items, item_matrix, n=10)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=comparacion, x='score_base', y='score_final', hue='nivel_alerta', palette='YlOrRd', s=90)
plt.title('Impacto de la penalidad climatica')
plt.show()
display(comparacion[['nombre', 'nivel_alerta', 'score_base', 'penalidad', 'score_final']])

## 6. Serializacion del modelo

In [ ]:
joblib.dump(item_matrix, MODELS_DIR / 'item_matrix.pkl')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(mlb, MODELS_DIR / 'mlb_servicios.pkl')
joblib.dump(ohe, MODELS_DIR / 'ohe_categoria.pkl')
df_items.to_parquet(MODELS_DIR / 'df_items.parquet', index=False)
print('Modelo serializado correctamente en:', MODELS_DIR)

## 7. Conclusiones

El EDA mostro que los establecimientos de primer nivel concentran la mayor parte de la oferta preventiva y que la cartera de servicios tiene una matriz dispersa, rasgo adecuado para filtrado basado en contenido. El cruce con alertas SENAMHI permitio reducir el score de establecimientos ubicados en zonas con heladas o precipitacion relevante. Los tres casos de prueba demostraron que el sistema prioriza cercania y servicio requerido, pero penaliza zonas climaticamente criticas. La serializacion final genera los componentes necesarios para reutilizar el modelo en el backend.